# AI-Based Automated Disease Risk Screening System



## 1. Import Libraries


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)



## 2. Locate and Load the Dataset


In [ ]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "dataset" / "heart_disease.csv").exists() and (PROJECT_DIR.parent / "dataset" / "heart_disease.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATASET_PATH = PROJECT_DIR / "dataset" / "heart_disease.csv"
OUTPUT_DATASET_PATH = PROJECT_DIR / "dataset" / "heart_disease_cleaned.csv"
MODEL_DIR = PROJECT_DIR / "model"
MODEL_PATH = MODEL_DIR / "heart_disease_model.pkl"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. Keep the dataset folder beside the notebooks folder."
    )

data = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully")
print("Dataset shape:", data.shape)
data.head()


## 3. Understand the Dataset


In [ ]:
print("Column names:")
print(data.columns.tolist())

print("\nDataset information:")
data.info()

print("\nDescriptive statistics:")
display(data.describe().T)

print("\nMissing values:")
display(data.isnull().sum().to_frame("Missing values"))

print("\nOriginal target distribution:")
print(data["target"].value_counts().sort_index())


## 4. Exploratory Data Analysis (EDA)


In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="target")
plt.title("Original Heart Disease Target Distribution")
plt.xlabel("Original target value")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

data.hist(figsize=(15, 12), bins=20, edgecolor="black")
plt.suptitle("Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 9))
sns.heatmap(data.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


## 5. Data Preprocessing


In [ ]:
clean_data = data.copy()

print("Missing values before preprocessing:")
display(clean_data.isnull().sum().to_frame("Missing values"))

for column in ["ca", "thal"]:
    clean_data[column] = clean_data[column].fillna(clean_data[column].median())

clean_data["target"] = (clean_data["target"] > 0).astype(int)

print("Missing values after preprocessing:")
display(clean_data.isnull().sum().to_frame("Missing values"))

print("Cleaned dataset shape:", clean_data.shape)
print("Binary target distribution:")
display(clean_data["target"].value_counts().sort_index().rename(index={0: "Class 0", 1: "Class 1"}))

OUTPUT_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_data.to_csv(OUTPUT_DATASET_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_DATASET_PATH}")
clean_data.head()


## 6. Separate Features and Target


In [ ]:
X = clean_data.drop(columns=["target"])
y = clean_data["target"]

feature_names = X.columns.tolist()
print("Number of features:", len(feature_names))
print("Features:", feature_names)
print("Target shape:", y.shape)


## 7. Train-Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target distribution:")
print(y_train.value_counts(normalize=True).sort_index())
print("Testing target distribution:")
print(y_test.value_counts(normalize=True).sort_index())


## 8. Train Machine Learning Models


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

trained_models = {}
predictions = {}

for model_name, classifier in models.items():
    classifier.fit(X_train, y_train)
    trained_models[model_name] = classifier
    predictions[model_name] = classifier.predict(X_test)
    print(f"{model_name} training completed.")


## 9. Evaluate and Compare Models


In [ ]:
comparison_rows = []
for model_name, y_pred in predictions.items():
    comparison_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
display(comparison_df.style.format("{:.4f}"))


In [ ]:
ax = comparison_df.plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=0)
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 10. Confusion Matrix and Classification Reports


In [ ]:
for model_name, y_pred in predictions.items():
    print(f"\n===== {model_name} =====")
    print(classification_report(y_test, y_pred, target_names=["Class 0", "Class 1"], zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Predicted 0", "Predicted 1"],
                yticklabels=["Actual 0", "Actual 1"])
    plt.title(f"Confusion Matrix — {model_name}")
    plt.xlabel("Prediction")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


## 11. Select and Save the Project Model


In [ ]:
best_model = trained_models["Random Forest"]
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
print(f"Random Forest model saved successfully: {MODEL_PATH}")


## 12. Check Predictions Against Test-Set Actual Labels


In [ ]:
test_predictions = best_model.predict(X_test)
result_table = X_test.copy()
result_table.insert(0, "Test Case", range(1, len(X_test) + 1))
result_table["Actual"] = y_test.to_numpy()
result_table["Predicted"] = test_predictions
result_table["Result"] = np.where(
    result_table["Actual"] == result_table["Predicted"], "CORRECT", "WRONG"
)

display(result_table[["Test Case", "Actual", "Predicted", "Result"]])
correct_count = int((y_test.to_numpy() == test_predictions).sum())
wrong_count = int((y_test.to_numpy() != test_predictions).sum())
print(f"Total test records: {len(y_test)}")
print(f"Correct predictions: {correct_count}")
print(f"Incorrect predictions: {wrong_count}")
print(f"Test accuracy: {accuracy_score(y_test, test_predictions):.2%}")


## 13. Try a Sample Prediction


In [ ]:
sample_patient = pd.DataFrame([{
    "age": 55,
    "sex": 1,
    "cp": 2,
    "trestbps": 130,
    "chol": 250,
    "fbs": 0,
    "restecg": 1,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 1.0,
    "slope": 1,
    "ca": 0,
    "thal": 2
}], columns=feature_names)

sample_prediction = best_model.predict(sample_patient)[0]
sample_class1_probability = best_model.predict_proba(sample_patient)[0][1]

print("Predicted dataset class:", int(sample_prediction))
print(f"Model-estimated probability for class 1: {sample_class1_probability:.2%}")
print("Reminder: educational screening output only; not a medical diagnosis.")
